# M-Shots Learning

In this notebook, we'll explore small prompt engineering techniques and recommendations that will help us elicit responses from the models that are better suited to our needs.

In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

# Formatting the answer with Few Shot Samples.

To obtain the model's response in a specific format, we have various options, but one of the most convenient is to use Few-Shot Samples. This involves presenting the model with pairs of user queries and example responses.

Large models like GPT-3.5 respond well to the examples provided, adapting their response to the specified format.

Depending on the number of examples given, this technique can be referred to as:
* Zero-Shot.
* One-Shot.
* Few-Shots.

With One Shot should be enough, and it is recommended to use a maximum of six shots. It's important to remember that this information is passed in each query and occupies space in the input prompt.



In [2]:
# Function to call the model.
def return_OAIResponse(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=1,
        )

    return (response.choices[0].message.content)

In this zero-shots prompt we obtain a correct response, but without formatting, as the model incorporates the information he wants.

In [3]:
#zero-shot
context_user = [
    {'role':'system', 'content':'You are an expert in F1.'}
]
print(return_OAIResponse("Who won the F1 2010?", context_user))

Sebastian Vettel won the F1 2010 championship driving for Red Bull Racing.


For a model as large and good as GPT 3.5, a single shot is enough to learn the output format we expect.


In [4]:
#one-shot
context_user = [
    {'role':'system', 'content':
     """You are an expert in F1.

     Who won the 2000 f1 championship?
     Driver: Michael Schumacher.
     Team: Ferrari."""}
]
print(return_OAIResponse("Who won the F1 2011?", context_user))

Driver: Sebastian Vettel.
Team: Red Bull Racing.


Smaller models, or more complicated formats, may require more than one shot. Here a sample with two shots.

In [5]:
#Few shots
context_user = [
    {'role':'system', 'content':
     """You are an expert in F1.

     Who won the 2010 f1 championship?
     Driver: Sebastian Vettel.
     Team: Red Bull Renault.

     Who won the 2009 f1 championship?
     Driver: Jenson Button.
     Team: BrawnGP."""}
]
print(return_OAIResponse("Who won the F1 2006?", context_user))

The 2006 Formula 1 World Championship was won by Fernando Alonso driving for Renault.


In [6]:
print(return_OAIResponse("Who won the F1 2019?", context_user))

The 2019 F1 World Championship was won by Lewis Hamilton driving for Mercedes.


We've been creating the prompt without using OpenAI's roles, and as we've seen, it worked correctly.

However, the proper way to do this is by using these roles to construct the prompt, making the model's learning process even more effective.

By not feeding it the entire prompt as if they were system commands, we enable the model to learn from a conversation, which is more realistic for it.

In [7]:
#Recomended solution
context_user = [
    {'role':'system', 'content':'You are and expert in f1.\n\n'},
    {'role':'user', 'content':'Who won the 2010 f1 championship?'},
    {'role':'assistant', 'content':"""Driver: Sebastian Vettel. \nTeam: Red Bull. \nPoints: 256. """},
    {'role':'user', 'content':'Who won the 2009 f1 championship?'},
    {'role':'assistant', 'content':"""Driver: Jenson Button. \nTeam: BrawnGP. \nPoints: 95. """},
]

print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton. 
Team: Mercedes. 
Points: 413.


We could also address it by using a more conventional prompt, describing what we want and how we want the format.

However, it's essential to understand that in this case, the model is following instructions, whereas in the case of use shots, it is learning in real-time during inference.

In [8]:
context_user = [
    {'role':'system', 'content':"""You are and expert in f1.
    You are going to answer the question of the user giving the name of the rider,
    the name of the team and the points of the champion, following the format:
    Drive:
    Team:
    Points: """
    }
]

print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton
Team: Mercedes
Points: 413


In [9]:
context_user = [
    {'role':'system', 'content':
     """You are classifying .

     Who won the 2010 f1 championship?
     Driver: Sebastian Vettel.
     Team: Red Bull Renault.

     Who won the 2009 f1 championship?
     Driver: Jenson Button.
     Team: BrawnGP."""}
]
print(return_OAIResponse("Who won the F1 2006?", context_user))

Driver: Fernando Alonso.
Team: Renault.


Few Shots for classification.


In [10]:
context_user = [
    {'role':'system', 'content':
     """You are an expert in reviewing product opinions and classifying them as positive or negative.

     It fulfilled its function perfectly, I think the price is fair, I would buy it again.
     Sentiment: Positive

     It didn't work bad, but I wouldn't buy it again, maybe it's a bit expensive for what it does.
     Sentiment: Negative.

     I wouldn't know what to say, my son uses it, but he doesn't love it.
     Sentiment: Neutral
     """}
]
print(return_OAIResponse("I'm not going to return it, but I don't plan to buy it again.", context_user))

Sentiment: Negative.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [11]:
# Version 1: Few-shot classification with emotions

context_user = [
    {'role':'system', 'content':
     """You classify short user messages into one emotion: Happy, Sad, Angry, or Neutral.

     I finally passed my exam!
     Emotion: Happy

     I can't believe they ignored my complaint again.
     Emotion: Angry

     Nothing special happened today.
     Emotion: Neutral

     I really miss my family.
     Emotion: Sad
     """}
]

print(return_OAIResponse("My package arrived broken and customer service was useless.", context_user))

Emotion: Angry


In [12]:
# Version 2: Few-shot travel recommendation format

context_user = [
    {'role':'system', 'content':
     """You are a travel assistant. Answer using this format:
     Destination:
     Reason:
     Best activity:

     I want a romantic weekend in Italy.
     Destination: Venice
     Reason: It is scenic, romantic, and walkable.
     Best activity: Gondola ride and dinner near the canals.

     I want a cheap beach trip in Spain.
     Destination: Valencia
     Reason: It has beaches, culture, and lower prices than Barcelona.
     Best activity: Visit the beach and the old town.
     """}
]

print(return_OAIResponse("I want a weekend trip with nature and hiking in Germany.", context_user))

Destination: Berchtesgaden National Park
Reason: It offers stunning natural landscapes, including mountains, lakes, and forests, perfect for hiking and immersing in nature.
Best activity: Hike to the top of Mount Watzmann for breathtaking panoramic views or explore the crystal clear waters of Lake Königssee.


In [13]:
# Version 3: Few-shot product review classification with explanation

context_user = [
    {'role':'system', 'content':
     """You classify product reviews as Positive, Negative, or Neutral.
     Also give a short reason.

     The headphones sound great and the battery lasts all day.
     Sentiment: Positive
     Reason: The user is satisfied with sound and battery.

     The product arrived late and stopped working after two days.
     Sentiment: Negative
     Reason: The user had delivery and quality problems.

     The bag is okay, not amazing, but usable.
     Sentiment: Neutral
     Reason: The user is neither clearly happy nor unhappy.
     """}
]

print(return_OAIResponse("The chair looks nice, but it is uncomfortable after one hour.", context_user))

Sentiment: Negative
Reason: The user finds the chair uncomfortable after one hour of use.


In this exercise, I tested different few-shot prompting techniques to control the model’s output format and behavior. The first version focused on emotion classification. By giving examples such as Happy, Angry, Neutral, and Sad, the model was able to classify a new sentence into the correct emotional category. This showed that few-shot prompting is useful when we want the model to follow a fixed label structure.

The second version used few-shot examples for travel recommendations. Instead of only answering normally, the model followed the exact format: Destination, Reason, and Best activity. This worked well because the examples clearly demonstrated the expected structure. I learned that examples are often more powerful than long explanations, especially when we want a specific output style.

The third version focused on product review sentiment classification with a short explanation. This version was useful because it did not only return a label, but also explained why the label was chosen. The model usually followed the format correctly, but sometimes the answer could be subjective, especially when the review contained both positive and negative information. For example, “looks nice but uncomfortable” could be classified as Negative or Neutral depending on how strongly the model interprets the complaint.

Some variations did not work as well when the prompt was unclear or when the examples were inconsistent. For example, if one example used “Sentiment: Positive” and another used only “Positive,” the model sometimes changed the format. Also, when there were too few examples for ambiguous cases, the model could give a less reliable answer.

Overall, I learned that few-shot prompting helps guide the model during inference without retraining it. Good examples should be clear, consistent, and close to the type of input we expect from the user. I also learned that using roles properly, such as system, user, and assistant messages, makes the prompt more realistic and easier for the model to follow.